# ALQAC 2026 — Kaggle Public Baseline

Notebook này clone private branch `TuanAnh`, chạy Public Test theo private-like boundary và tạo `submission.json`. Mặc định chạy smoke test 2 cases bằng `baseline.yaml` (Case API + BM25 + Qwen3-8B 4-bit). Notebook không upload leaderboard.

## 0. Kaggle prerequisites

Trước khi Run All:

1. Notebook Settings → Accelerator → GPU T4.
2. Notebook Settings → Internet → On.
3. Add-ons → Secrets → thêm `GITHUB_TOKEN` có quyền read private repo.
4. Add-ons → Secrets → thêm `ALQAC_TEAM_TOKEN` do BTC cung cấp.
5. Bật quyền truy cập cả hai secret cho notebook.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import tempfile

IS_KAGGLE = Path('/kaggle').exists()
assert IS_KAGGLE, 'Notebook này được chuẩn bị cho Kaggle runtime'

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
github_token = secrets.get_secret('GITHUB_TOKEN')
alqac_token = secrets.get_secret('ALQAC_TEAM_TOKEN')
assert github_token, 'Missing Kaggle Secret: GITHUB_TOKEN'
assert alqac_token, 'Missing Kaggle Secret: ALQAC_TEAM_TOKEN'
os.environ['ALQAC_TEAM_TOKEN'] = alqac_token
print('Kaggle secrets loaded (values are not displayed).')

## 1. Clone/update đúng private branch

Authentication dùng temporary `GIT_ASKPASS`; token không được đặt trong URL, source hoặc output.

In [ ]:
REPO_URL = 'https://github.com/NGBao1608/DL_K23_ALQAC2026.git'
BRANCH = 'TuanAnh'
PROJECT_ROOT = Path('/kaggle/working/DL_K23_ALQAC2026')

with tempfile.TemporaryDirectory() as temporary_dir:
    askpass = Path(temporary_dir) / 'git-askpass.sh'
    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) echo "x-access-token" ;;\n'
        '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
        'esac\n',
        encoding='utf-8',
    )
    askpass.chmod(0o700)
    git_env = os.environ.copy()
    git_env.update({
        'GITHUB_TOKEN': github_token,
        'GIT_ASKPASS': str(askpass),
        'GIT_TERMINAL_PROMPT': '0',
    })
    if not (PROJECT_ROOT / '.git').exists():
        subprocess.run(
            ['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(PROJECT_ROOT)],
            env=git_env,
            check=True,
        )
    else:
        subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=PROJECT_ROOT, env=git_env, check=True)
        subprocess.run(['git', 'checkout', BRANCH], cwd=PROJECT_ROOT, check=True)
        subprocess.run(['git', 'merge', '--ff-only', f'origin/{BRANCH}'], cwd=PROJECT_ROOT, check=True)

current_branch = subprocess.check_output(
    ['git', 'branch', '--show-current'], cwd=PROJECT_ROOT, text=True
).strip()
current_commit = subprocess.check_output(
    ['git', 'rev-parse', '--short', 'HEAD'], cwd=PROJECT_ROOT, text=True
).strip()
assert current_branch == BRANCH, f'Expected {BRANCH}, got {current_branch}'
assert (PROJECT_ROOT / 'pyproject.toml').exists()
assert (PROJECT_ROOT / 'data/raw/ALQAC2026_public_test.json').exists()
assert (PROJECT_ROOT / 'data/raw/corpus_law_pub.json').exists()
print({'branch': current_branch, 'commit': current_commit, 'root': str(PROJECT_ROOT)})

## 2. Install package và kiểm tra GPU

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_ROOT)],
    check=True,
)

import torch

assert torch.cuda.is_available(), 'GPU chưa được bật trong Kaggle Settings'
print({'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__})

## 3. Chọn experiment

Giữ `RUN_MODE='smoke'` trong lần đầu. Chỉ đổi thành `'full'` sau khi smoke test có `validation.status == 'PASS'`. Smoke và full luôn dùng hai run directory khác nhau.

In [ ]:
EXPERIMENT = 'baseline'  # baseline | candidate
RUN_MODE = 'smoke'       # smoke | full

assert EXPERIMENT in {'baseline', 'candidate'}
assert RUN_MODE in {'smoke', 'full'}
CONFIG_PATH = PROJECT_ROOT / f'configs/{EXPERIMENT}.yaml'
RUN_DIR = Path('/kaggle/working/alqac2026/outputs') / f'public_{EXPERIMENT}_{RUN_MODE}'
LIMIT = 2 if RUN_MODE == 'smoke' else None

print({
    'experiment': EXPERIMENT,
    'mode': RUN_MODE,
    'limit': LIMIT,
    'config': str(CONFIG_PATH),
    'run_dir': str(RUN_DIR),
})

## 4. Chạy pipeline

Nếu Kaggle session bị ngắt, chạy lại notebook với cùng `EXPERIMENT`, `RUN_MODE`, branch và source commit để resume checkpoint. Không dùng smoke run directory cho full run.

In [ ]:
command = [
    sys.executable,
    str(PROJECT_ROOT / 'scripts/run_public.py'),
    '--config', str(CONFIG_PATH.relative_to(PROJECT_ROOT)),
    '--input', 'data/raw/ALQAC2026_public_test.json',
    '--resume-run', str(RUN_DIR),
]
if LIMIT is not None:
    command.extend(['--limit', str(LIMIT)])

subprocess.run(command, cwd=PROJECT_ROOT, env=os.environ.copy(), check=True)

## 5. Validate và xem metrics

In [ ]:
def read_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

manifest = read_json(RUN_DIR / 'manifest.json')
validation = read_json(RUN_DIR / 'validation.json')
metrics = read_json(RUN_DIR / 'metrics.json')
api_stats = read_json(RUN_DIR / 'api_stats.json')
expected_cases = 2 if RUN_MODE == 'smoke' else 50

assert manifest['run']['status'] == 'completed', manifest['run']
assert manifest['run']['completed'] == expected_cases, manifest['run']
assert validation['status'] == 'PASS', validation
assert validation['cases'] == expected_cases, validation

print('VALIDATION:', validation)
print('METRICS:')
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print('API STATS:')
print(json.dumps(api_stats, ensure_ascii=False, indent=2))

## 6. Đóng gói artifact để tải từ Kaggle

Không upload file smoke lên leaderboard. Với full run, file cần nộp là `submission.json`; ZIP dưới đây giữ thêm validation, metrics và manifest để kiểm chứng.

In [ ]:
EXPORT_DIR = Path('/kaggle/working/export') / f'public_{EXPERIMENT}_{RUN_MODE}'
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
EXPORT_DIR.mkdir(parents=True)

for filename in ('submission.json', 'validation.json', 'metrics.json', 'manifest.json', 'api_stats.json'):
    shutil.copy2(RUN_DIR / filename, EXPORT_DIR / filename)

archive_path = shutil.make_archive(str(EXPORT_DIR), 'zip', root_dir=EXPORT_DIR)
print('Submission:', RUN_DIR / 'submission.json')
print('Download archive:', archive_path)